# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023.

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**

### Make relevant library imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
# Load raw data
df = pd.read_csv('data/AviationData.csv', encoding='latin-1', low_memory=False)

print('Shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
df.head()

In [ ]:
# Inspect data types
df.dtypes

In [ ]:
# Inspect missing values
df.isnull().sum()

In [ ]:
# Summary statistics for numeric columns
df.describe()

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in:
- Only Accidents (not incidents)
- From 1983 onwards (max 40-year lifetime)
- United States only
- Professional builds only (Amateur.Built == 'No')
- Airplanes only (Aircraft.Category == 'Airplane')

In [ ]:
# Convert Event.Date to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter: Accidents only
df = df[df['Investigation.Type'] == 'Accident']

# Filter: 1983 onwards
df = df[df['Event.Date'].dt.year >= 1983]

# Filter: United States only
df = df[df['Country'] == 'United States']

# Filter: Professional builds only
df = df[df['Amateur.Built'] == 'No']

# Filter: Airplanes only
df = df[df['Aircraft.Category'] == 'Airplane']

print('Shape after filtering:', df.shape)
df.head()

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. We will clean and impute relevant columns and then create derived fields that best quantify what the client wishes to track.

**Construct metric for fatal/serious injuries**

We estimate the total number of passengers on each flight by summing all injury categories (fatal + serious + minor + uninjured). The likelihood of serious/fatal injury is then estimated as a fraction of this total.

In [ ]:
# Fill NaNs in injury columns with 0
# Assumption: missing injury counts mean no injuries in that category
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 
               'Total.Minor.Injuries', 'Total.Uninjured']

df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total passengers on board
df['Total.Passengers'] = (df['Total.Fatal.Injuries'] + 
                          df['Total.Serious.Injuries'] + 
                          df['Total.Minor.Injuries'] + 
                          df['Total.Uninjured'])

# Create fraction of fatal/serious injuries
# Where total passengers is 0, set fraction to 0 to avoid division by zero
df['Fatal.Serious.Fraction'] = np.where(
    df['Total.Passengers'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers'],
    0
)

print(df[['Total.Passengers', 'Fatal.Serious.Fraction']].describe())
df[['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Passengers', 'Fatal.Serious.Fraction']].head()

**Aircraft.Damage**
- Identify and execute cleaning tasks
- Construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# Inspect Aircraft.damage column
print(df['Aircraft.damage'].value_counts())
print('NaNs:', df['Aircraft.damage'].isna().sum())

In [ ]:
# Standardize casing
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Drop rows where Aircraft.damage is NaN or Unknown - we cannot assess destruction risk
df = df[df['Aircraft.damage'].notna()]
df = df[df['Aircraft.damage'] != 'Unknown']

# Create binary destroyed column: 1 if Destroyed, 0 otherwise
df['Destroyed'] = (df['Aircraft.damage'] == 'Destroyed').astype(int)

print(df['Destroyed'].value_counts())
print('Shape after damage cleaning:', df.shape)

### Investigate the *Make* column

**Cleaning tasks identified:**
1. Inconsistent casing (e.g. 'CESSNA' vs 'Cessna' vs 'cessna') — standardize with `.str.title()`
2. Leading/trailing whitespace — fix with `.str.strip()`
3. Drop rows with missing Make
4. Keep only Makes with at least 50 records for statistical robustness

In [ ]:
# Inspect Make column before cleaning
print('NaNs in Make:', df['Make'].isna().sum())
print('\nTop makes before cleaning:')
print(df['Make'].value_counts().head(20))

In [ ]:
# Drop rows with missing Make
df = df[df['Make'].notna()]

# Standardize casing and strip whitespace
df['Make'] = df['Make'].str.strip().str.title()

print('Top makes after cleaning:')
print(df['Make'].value_counts().head(20))

In [ ]:
# Keep only Makes with at least 50 records
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print(f'Number of valid Makes: {len(valid_makes)}')
print(f'Shape after Make filtering: {df.shape}')

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- Create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Drop rows with missing Model
print('NaNs in Model:', df['Model'].isna().sum())
df = df[df['Model'].notna()]

# Standardize Model
df['Model'] = df['Model'].str.strip().str.upper()

print('\nSample model counts:')
print(df['Model'].value_counts().head(10))

In [ ]:
# Check if model labels are unique to each make
# A model name appearing under multiple makes means it's not unique
model_make_counts = df.groupby('Model')['Make'].nunique()
print('Models shared across multiple makes:')
print(model_make_counts[model_make_counts > 1].head(10))

# Create unique Make_Model identifier
df['Make_Model'] = df['Make'] + ' ' + df['Model']

print('\nSample Make_Model values:')
print(df['Make_Model'].value_counts().head(10))

### Cleaning other columns
Inspect and clean: Engine.Type, Weather.Condition, Number.of.Engines, Purpose.of.flight, Broad.phase.of.flight

In [ ]:
# Engine.Type
print('Engine.Type value counts:')
print(df['Engine.Type'].value_counts())
print('NaNs:', df['Engine.Type'].isna().sum())

In [ ]:
# Standardize Engine.Type
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()
print(df['Engine.Type'].value_counts())

In [ ]:
# Weather.Condition
print('Weather.Condition value counts:')
print(df['Weather.Condition'].value_counts())

# Standardize
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()
print(df['Weather.Condition'].value_counts())

In [ ]:
# Number.of.Engines
print('Number.of.Engines value counts:')
print(df['Number.of.Engines'].value_counts())

In [ ]:
# Purpose.of.flight
print('Purpose.of.flight value counts:')
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()
print(df['Purpose.of.flight'].value_counts())

In [ ]:
# Broad.phase.of.flight
print('Broad.phase.of.flight value counts:')
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()
print(df['Broad.phase.of.flight'].value_counts())

### Column Removal
- Inspect the dataframe and drop any columns that have too many NaNs (>60% missing)

In [ ]:
# Calculate percentage of missing values per column
missing_pct = df.isnull().mean() * 100
print('Missing % per column:')
print(missing_pct.sort_values(ascending=False))

# Drop columns with more than 60% missing values
cols_to_drop = missing_pct[missing_pct > 60].index.tolist()
print('\nColumns to drop:', cols_to_drop)
df = df.drop(columns=cols_to_drop)

print('\nShape after column removal:', df.shape)

### Save DataFrame to csv
Save the cleaned data so it can be loaded directly in the analysis notebook.

In [ ]:
df.to_csv('data/AviationData_Cleaned.csv', index=False)
print('Cleaned data saved!')
print('Final shape:', df.shape)
df.head()